In [6]:
import pandas as pd
import numpy as np
import seaborn as sns
import unidecode as unidecode
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from tabfm import TabFMClassifier
from tabfm import tabfm_v1_0_0_pytorch as tabfm_v1_0_0
import torch
from google.colab import drive

drive.mount("/content/drive")

df1 = pd.read_csv("/content/drive/MyDrive/DADOS_CRIMINAIS_SP/JAN_JUN_2025.csv", encoding='utf-8', sep=';')
df2 = pd.read_csv("/content/drive/MyDrive/DADOS_CRIMINAIS_SP/JUL_DEZ_2025.csv", encoding='utf-8', sep=';')
full_df = pd.concat([df1, df2], ignore_index=True)

# FERIADOS = [01/01, 25/01, 03/03, 04/03, 05/03, 01/04, 18/4, 20/4, 21/04, 01/05, 11/05, 12/06, 09/07, 10/08, 07/09, 12/10, 15/10, 17/10,
# 28/10, 02/11, 15/11, 20/11, 25/12 - dia dos namorados não é feriado, porém foi incluso para testar a teoria de aumento de crimes passionais
holidays_sp = {
    (1,1),   # Ano novo
    (3,3),   # Carnaval
    (3,4),   # Carnaval
    (3,5),   # CarnavaL
    (4,18),  # Sexta-feira santa
    (4,20),  # Páscoa
    (4,21),  # Tiradentes
    (5,1),   # Dia do trabalho
    (5,11),  # Dia das mãeslap
    (6,12),  # Dia dos namorados
    (6,19),  # Corpus Chisti
    (7,9),   # Revolução Constitucionalista
    (8,10),  # Dia dos pais
    (9,7),   # Independência do Brasil
    (10,12), # Nossa Senhora Aparecida
    (10,15), # Dia do Professor
    (10,17), # Dia do Comércio
    (10,28), # Dia do Servidor Público
    (11,2),  # Finados
    (11,15), # Proclamação da República
    (11,20), # Consciência Negra
    (12,25), # Natal
  }

df_filtered = full_df.drop(columns=["ANO_BO","NOME_DEPARTAMENTO", "NOME_SECCIONAL", "NOME_DELEGACIA", "NUM_BO", "DATA_REGISTRO",
                                    "DESCR_SUBTIPOLOCAL", "LOGRADOURO","NUMERO_LOGRADOURO", "LATITUDE", "LONGITUDE",
                                    "NOME_DELEGACIA_CIRCUNSCRICAO", "NOME_DEPARTAMENTO_CIRCUNSCRICAO", "NOME_SECCIONAL_CIRCUNSCRICAO",
                                    "NOME_MUNICIPIO_CIRCUNSCRICAO", "DESCR_CONDUTA", "NATUREZA_APURADA", "MES_ESTATISTICA",
                                    "ANO_ESTATISTICA", "CMD", "BTL", "CIA"])

# Transforma tudo de maiúsculo para minúsculo
df_lowercase = df_filtered.apply(lambda x: x.str.lower())

# Remove todas pontuações
columns_clean = ['NOME_MUNICIPIO', 'DESC_PERIODO','BAIRRO', 'DESCR_TIPOLOCAL', 'RUBRICA']
for col in columns_clean:
    df_lowercase[col] = df_lowercase[col].map(lambda x: unidecode.unidecode(x) if isinstance(x, str) else x)

# Conversão e criação de features temporais

def create_time_features(df_lowercase, time_feature='DATA_OCORRENCIA_BO'):
  df_lowercase[time_feature] = pd.to_datetime(df_lowercase[time_feature], dayfirst=True, errors='coerce')

  df_lowercase['ANO'] = df_lowercase[time_feature].dt.year.astype('Int64')
  df_lowercase['MES'] = df_lowercase[time_feature].dt.month.astype('Int64')
  df_lowercase['DIA'] = df_lowercase[time_feature].dt.day.astype('Int64')
  df_lowercase['DIA_SEM'] = df_lowercase[time_feature].dt.day_name().str.lower()
  df_lowercase['FERIADO'] = False

  return df_lowercase
df_lowercase = create_time_features(df_lowercase)

# Remoção de nulos
df_no_nulls = (df_lowercase.dropna(subset=["DESC_PERIODO", "HORA_OCORRENCIA_BO"], how="all").drop(columns=["DATA_OCORRENCIA_BO"]).dropna(subset=['DESCR_TIPOLOCAL', 'BAIRRO', 'RUBRICA', 'ANO', 'MES', 'DIA_SEM', 'DIA']))
df_in_scope = df_no_nulls[(df_no_nulls['DESC_PERIODO'] != 'em hora incerta') & (df_no_nulls['ANO'] == 2025)]

# Aplicação de masks que preenchem valores nulos da coluna 'DESC_PERIODO' com base nos horários de 'HORA_OCORRENCIA_BO'
mask_morning = df_in_scope['HORA_OCORRENCIA_BO'].between('06:00:00', '11:59:59')
mask_afternoon = df_in_scope['HORA_OCORRENCIA_BO'].between('12:00:00', '17:59:59')
mask_night = df_in_scope['HORA_OCORRENCIA_BO'].between('18:00:00', '23:59:59')
mask_dawn = df_in_scope['HORA_OCORRENCIA_BO'].between('00:00:00', '05:59:59')

df_masked = df_in_scope.copy()

df_masked.loc[mask_morning, 'DESC_PERIODO'] = "pela manha"
df_masked.loc[mask_afternoon, 'DESC_PERIODO'] = "a tarde"
df_masked.loc[mask_night, 'DESC_PERIODO'] = "a noite"
df_masked.loc[mask_dawn, 'DESC_PERIODO'] = "de madrugada"

# Associa dia com feriado
def holiday(dia, mes):
  return (mes, dia) in holidays_sp

df_masked['FERIADO'] =df_masked.apply(lambda row: holiday(row['DIA'], row['MES']), axis = 1)

# Configuração do modelo
model = tabfm_v1_0_0.load(model_type="classification")
tabfm = TabFMClassifier(model=model, random_state=42, n_estimators = 1, max_num_rows=100)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


config.json:   0%|          | 0.00/97.0 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights from local directory


In [7]:
df_masked.head()

,NOME_MUNICIPIO,HORA_OCORRENCIA_BO,DESC_PERIODO,DESCR_TIPOLOCAL,BAIRRO,RUBRICA,ANO,MES,DIA,DIA_SEM,FERIADO
1,s.paulo,00:00:00,de madrugada,via publica,se,estupro de vulneravel (art.217-a),2025,1,10,friday,False
3,s.paulo,18:00:14,a noite,hospedagem,caninde,estupro de vulneravel (art.217-a),2025,1,25,saturday,False
9,s.paulo,NaN,de madrugada,via publica,liberdade,furto (art. 155),2025,1,1,wednesday,True
11,s.paulo,NaN,a tarde,via publica,se,furto (art. 155),2025,1,1,wednesday,True
12,s.paulo,NaN,a tarde,via publica,se,furto (art. 155),2025,1,1,wednesday,True


In [33]:
print(f"Itupeva: {len(df_masked[df_masked['NOME_MUNICIPIO'] == 'itupeva'])} instâncias")
print(f"Sertaozinho: {len(df_masked[df_masked['NOME_MUNICIPIO'] == 'sertaozinho'])} instâncias")
print(f"Taubate: {len(df_masked[df_masked['NOME_MUNICIPIO'] == 'taubate'])} instâncias")
print(f"Pindamonhangaba: {len(df_masked[df_masked['NOME_MUNICIPIO'] == 'pindamonhangaba'])} instâncias")
print(f"Birigui: {len(df_masked[df_masked['NOME_MUNICIPIO'] == 'birigui'])} instâncias")

Itupeva: 1073 instâncias
Sertaozinho: 2087 instâncias
Taubate: 5801 instâncias
Pindamonhangaba: 2438 instâncias
Birigui: 2097 instâncias


In [ ]:
# TEST: itupeva
# TRAIN: sertaozinho, taubate, pindamonhangaba, birigui

df_train = df_rem4[df_rem4['NOME_MUNICIPIO'].isin(['sertaozinho', 'taubate', 'pindamonhangaba', 'birigui'])]
df_teste = df_rem4[df_rem4['NOME_MUNICIPIO'] == 'itupeva']

features = ['MES', 'DIA_SEM', 'FERIADO', 'RUBRICA', 'DESCR_TIPOLOCAL']

Xtrain = df_train[features].copy()
ytrain = df_train['DESC_PERIODO'].copy()

Xtest = df_teste[features].copy()
ytest = df_teste['DESC_PERIODO'].copy()

categorical_cols = ['DIA_SEM', 'FERIADO', 'RUBRICA', 'DESCR_TIPOLOCAL']

for col in categorical_cols:
    Xtrain[col] = Xtrain[col].astype(str)
    Xtest[col] = Xtest[col].astype(str)

Xtrain['MES'] = Xtrain['MES'].astype(float)
Xtest['MES'] = Xtest['MES'].astype(float)

In [ ]:
print("Treino:", Xtrain.shape)
print("Teste:", Xtest.shape)
print("\nTreino:")
print(ytrain.value_counts())
print("\nTeste:")

print(ytest.value_counts())

tabfm.fit(Xtrain, ytrain.to_numpy())
ypred = tabfm.predict(Xtest)
print(classification_report(ytest, ypred, zero_division=0))

In [ ]:
# Treino: (12423, 5)
# Teste: (1073, 5)

# Treino:
# DESC_PERIODO
# a noite         3469
# a tarde         3409
# pela manha      3016
# de madrugada    2529
# Name: count, dtype: int64

# Teste:
# DESC_PERIODO
# a tarde         365
# a noite         326
# pela manha      227
# de madrugada    155
# Name: count, dtype: int64
#               precision    recall  f1-score   support

#      a noite       0.34      0.40      0.37       326
#      a tarde       0.37      0.55      0.44       365
# de madrugada       0.20      0.12      0.15       155
#   pela manha       0.18      0.04      0.06       227

#     accuracy                           0.34      1073
#    macro avg       0.27      0.28      0.26      1073
# weighted avg       0.30      0.34      0.30      1073

In [ ]:
# TEST: sertaozinho
# TRAIN: itupeva, taubate, pindamonhangaba, birigui

df_train2 = df_rem4[df_rem4['NOME_MUNICIPIO'].isin(['itupeva', 'taubate', 'pindamonhangaba', 'birigui'])]
df_teste2 = df_rem4[df_rem4['NOME_MUNICIPIO'] == 'sertaozinho']

features2 = ['MES', 'DIA_SEM', 'FERIADO', 'RUBRICA', 'DESCR_TIPOLOCAL']

Xtrain = df_train2[features2].copy()
ytrain = df_train2['DESC_PERIODO'].copy()

Xtest = df_teste2[features2].copy()
ytest = df_teste2['DESC_PERIODO'].copy()

categorical_cols = ['DIA_SEM', 'FERIADO', 'RUBRICA', 'DESCR_TIPOLOCAL']

for col in categorical_cols:
    Xtrain[col] = Xtrain[col].astype(str)
    Xtest[col] = Xtest[col].astype(str)

Xtrain['MES'] = Xtrain['MES'].astype(float)
Xtest['MES'] = Xtest['MES'].astype(float)

In [ ]:
print("Treino:", Xtrain.shape)
print("Teste:", Xtest.shape)
print("\nTreino:")
print(ytrain.value_counts())
print("\nTeste:")
print(ytest.value_counts())

tabfm.fit(Xtrain, ytrain.to_numpy())
ypred = tabfm.predict(Xtest)
print(classification_report(ytest, ypred, zero_division=0))

In [ ]:
# Treino: (11409, 5)
# Teste: (2087, 5)

# Treino:
# DESC_PERIODO
# a tarde         3175
# a noite         3167
# pela manha      2672
# de madrugada    2395
# Name: count, dtype: int64

# Teste:
# DESC_PERIODO
# a noite         628
# a tarde         599
# pela manha      571
# de madrugada    289
# Name: count, dtype: int64
#               precision    recall  f1-score   support

#      a noite       0.31      0.26      0.28       628
#      a tarde       0.23      0.09      0.13       599
# de madrugada       0.24      0.21      0.23       289
#   pela manha       0.30      0.55      0.38       571

#     accuracy                           0.28      2087
#    macro avg       0.27      0.28      0.26      2087
# weighted avg       0.27      0.28      0.26      2087

# 12 minutos cpu
# 5 minutos gpu

In [ ]:
# TEST: taubate
# TRAIN: itupeva, sertaozinho, pindamonhangaba, birigui

df_train3 = df_rem4[df_rem4['NOME_MUNICIPIO'].isin(['itupeva', 'sertaozinho', 'pindamonhangaba', 'birigui'])]
df_teste3 = df_rem4[df_rem4['NOME_MUNICIPIO'] == 'taubate']

features3 = ['MES', 'DIA_SEM', 'FERIADO', 'RUBRICA', 'DESCR_TIPOLOCAL']

Xtrain = df_train3[features3].copy()
ytrain = df_train3['DESC_PERIODO'].copy()

Xtest = df_teste3[features3].copy()
ytest = df_teste3['DESC_PERIODO'].copy()

categorical_cols = ['DIA_SEM', 'FERIADO', 'RUBRICA', 'DESCR_TIPOLOCAL']

for col in categorical_cols:
    Xtrain[col] = Xtrain[col].astype(str)
    Xtest[col] = Xtest[col].astype(str)

Xtrain['MES'] = Xtrain['MES'].astype(float)
Xtest['MES'] = Xtest['MES'].astype(float)

In [ ]:
print("Treino:", Xtrain.shape)
print("Teste:", Xtest.shape)
print("\nTreino:")
print(ytrain.value_counts())
print("\nTeste:")
print(ytest.value_counts())

tabfm.fit(Xtrain, ytrain.to_numpy())
ypred = tabfm.predict(Xtest)
print(classification_report(ytest, ypred, zero_division=0))

In [ ]:
# Treino: (7695, 5)
# Teste: (5801, 5)

# Treino:
# DESC_PERIODO
# a noite         2234
# a tarde         2225
# pela manha      1942
# de madrugada    1294
# Name: count, dtype: int64

# Teste:
# DESC_PERIODO
# a noite         1561
# a tarde         1549
# de madrugada    1390
# pela manha      1301
# Name: count, dtype: int64
#               precision    recall  f1-score   support

#      a noite       0.29      0.75      0.42      1561
#      a tarde       0.26      0.26      0.26      1549
# de madrugada       0.28      0.03      0.06      1390
#   pela manha       0.22      0.02      0.03      1301

#     accuracy                           0.28      5801
#    macro avg       0.26      0.26      0.19      5801
# weighted avg       0.26      0.28      0.20      5801

# 42 minutos cpu
# 16 minutos gpu

In [ ]:
# TEST: pindamonhangaba
# TRAIN: taubate, itupeva, sertaozinho, birigui

df_train4 = df_rem4[df_rem4['NOME_MUNICIPIO'].isin(['taubate', 'itupeva', 'sertaozinho', 'birigui'])]
df_teste4 = df_rem4[df_rem4['NOME_MUNICIPIO'] == 'pindamonhangaba']

features4 = ['MES', 'DIA_SEM', 'FERIADO', 'RUBRICA', 'DESCR_TIPOLOCAL']

Xtrain = df_train4[features4].copy()
ytrain = df_train4['DESC_PERIODO'].copy()

Xtest = df_teste4[features4].copy()
ytest = df_teste4['DESC_PERIODO'].copy()

categorical_cols = ['DIA_SEM', 'FERIADO', 'RUBRICA', 'DESCR_TIPOLOCAL']

for col in categorical_cols:
    Xtrain[col] = Xtrain[col].astype(str)
    Xtest[col] = Xtest[col].astype(str)

Xtrain['MES'] = Xtrain['MES'].astype(float)
Xtest['MES'] = Xtest['MES'].astype(float)

In [ ]:
print("Treino:", Xtrain.shape)
print("Teste:", Xtest.shape)
print("\nTreino:")
print(ytrain.value_counts())
print("\nTeste:")
print(ytest.value_counts())

tabfm.fit(Xtrain, ytrain.to_numpy())
ypred = tabfm.predict(Xtest)
print(classification_report(ytest, ypred, zero_division=0))

In [ ]:
# TEST: birigui
# TRAIN: itupeva, sertaozinho, taubate, pindamonhangaba
df_train5 = df_rem4[df_rem4['NOME_MUNICIPIO'].isin(['itupeva', 'sertaozinho', 'taubate', 'pindamonhangaba'])]
df_teste5 = df_rem4[df_rem4['NOME_MUNICIPIO'] == 'birigui']

features5 = ['MES', 'DIA_SEM', 'FERIADO', 'RUBRICA', 'DESCR_TIPOLOCAL']

Xtrain = df_train5[features5].copy()
ytrain = df_train5['DESC_PERIODO'].copy()

Xtest = df_teste5[features5].copy()
ytest = df_teste5['DESC_PERIODO'].copy()

categorical_cols = ['DIA_SEM', 'FERIADO', 'RUBRICA', 'DESCR_TIPOLOCAL']

for col in categorical_cols:
    Xtrain[col] = Xtrain[col].astype(str)
    Xtest[col] = Xtest[col].astype(str)

Xtrain['MES'] = Xtrain['MES'].astype(float)
Xtest['MES'] = Xtest['MES'].astype(float)

In [ ]:
print("Treino:", Xtrain.shape)
print("Teste:", Xtest.shape)
print("\nTreino:")
print(ytrain.value_counts())
print("\nTeste:")
print(ytest.value_counts())

tabfm.fit(Xtrain, ytrain.to_numpy())
ypred = tabfm.predict(Xtest)
print(classification_report(ytest, ypred, zero_division=0))

In [ ]:
# 13 minutos cpu
# 5 minutos gpu



# Treino: (11399, 5)
# Teste: (2097, 5)

# Treino:
# DESC_PERIODO
# a noite         3225
# a tarde         3154
# pela manha      2666
# de madrugada    2354
# Name: count, dtype: int64

# Teste:
# DESC_PERIODO
# a tarde         620
# pela manha      577
# a noite         570
# de madrugada    330
# Name: count, dtype: int64
#               precision    recall  f1-score   support

#      a noite       0.29      0.07      0.11       570
#      a tarde       0.30      0.59      0.40       620
# de madrugada       0.23      0.38      0.29       330
#   pela manha       0.29      0.11      0.16       577

#     accuracy                           0.28      2097
#    macro avg       0.28      0.29      0.24      2097
# weighted avg       0.28      0.28      0.24      2097